# EMNLP 2025 Corpus Mapping — Step 1: Audit and Cleaning

This notebook audits and cleans the EMNLP 2025 paper dataset before semantic embedding, clustering, and research-gap analysis.

Expected input file:
`data/emnlp_2025_main.xlsx`


In [1]:
import pandas as pd
from pathlib import Path

# File path
DATA_FILE = Path("../data/emnlp_2025_main.xlsx")

# Load dataset
df = pd.read_excel(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()


Dataset loaded successfully.
Shape: (1809, 6)


,paper_id,title,authors,abstract,paper_url,pdf_url
0,2025.emnlp-main.1,Towards Automated Error Discovery: A Study in ...,Dominic Petrak; Thy Thy Tran; Iryna Gurevych,Although LLM-based conversational agents demon...,https://aclanthology.org/2025.emnlp-main.1/,https://aclanthology.org/2025.emnlp-main.1.pdf
1,2025.emnlp-main.2,Break the Checkbox: Challenging Closed-Style E...,Mohsinul Kabir; Ajwad Abrar; Sophia Ananiadou,A large number of studies rely on closed-style...,https://aclanthology.org/2025.emnlp-main.2/,https://aclanthology.org/2025.emnlp-main.2.pdf
2,2025.emnlp-main.3,Biased Tales: Cultural and Topic Bias in Gener...,Donya Rooein; Vilém Zouhar; Debora Nozza; Dirk...,Stories play a pivotal role in human communica...,https://aclanthology.org/2025.emnlp-main.3/,https://aclanthology.org/2025.emnlp-main.3.pdf
3,2025.emnlp-main.4,Large Language Models as Realistic Microservic...,Donghyun Kim; Sriram Ravula; Taemin Ha; Alex D...,Workload traces are essential to understand co...,https://aclanthology.org/2025.emnlp-main.4/,https://aclanthology.org/2025.emnlp-main.4.pdf
4,2025.emnlp-main.5,JUDGEBERT : Assessing Legal Meaning Preservati...,David Beauchemin; Michelle Albert-Rochette; Ri...,Simplifying text while preserving its meaning ...,https://aclanthology.org/2025.emnlp-main.5/,https://aclanthology.org/2025.emnlp-main.5.pdf


## 1. Inspect dataset structure

Check the available columns, data types, and missing values.

In [2]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())


Columns:
['paper_id', 'title', 'authors', 'abstract', 'paper_url', 'pdf_url']

Data types:
paper_id     str
title        str
authors      str
abstract     str
paper_url    str
pdf_url      str
dtype: object

Missing values:
paper_id     0
title        0
authors      0
abstract     0
paper_url    0
pdf_url      0
dtype: int64


## 2. Check for duplicate papers

Duplicate paper IDs or titles may indicate repeated records.

In [3]:
print("Duplicate paper IDs:", df["paper_id"].duplicated().sum())
print("Duplicate titles:", df["title"].duplicated().sum())


Duplicate paper IDs: 0
Duplicate titles: 0


## 3. Detect missing or empty abstracts

Some spreadsheet cells may contain blank strings rather than true missing values.

In [4]:
df["abstract"] = df["abstract"].replace(r"^\s*$", pd.NA, regex=True)

print("Missing abstracts:", df["abstract"].isna().sum())


Missing abstracts: 0


## 4. Inspect abstract lengths

Very short abstracts can indicate incomplete scraping or malformed records.

In [5]:
df["abstract_word_count"] = (
    df["abstract"]
    .fillna("")
    .str.split()
    .str.len()
)

df["abstract_word_count"].describe()


count    1809.000000
mean      159.655611
std        30.928536
min        46.000000
25%       139.000000
50%       159.000000
75%       179.000000
max       269.000000
Name: abstract_word_count, dtype: float64

In [6]:
df[df["abstract_word_count"] < 20][
    ["paper_id", "title", "abstract", "abstract_word_count"]
].head(20)


,paper_id,title,abstract,abstract_word_count


## 5. Create combined text field

This combines each paper's title and abstract. The resulting `text` column will be used later for semantic embeddings.

In [7]:
df["text"] = (
    df["title"].fillna("").str.strip()
    + ". "
    + df["abstract"].fillna("").str.strip()
)

df[["paper_id", "title", "text"]].head()


,paper_id,title,text
0,2025.emnlp-main.1,Towards Automated Error Discovery: A Study in ...,Towards Automated Error Discovery: A Study in ...
1,2025.emnlp-main.2,Break the Checkbox: Challenging Closed-Style E...,Break the Checkbox: Challenging Closed-Style E...
2,2025.emnlp-main.3,Biased Tales: Cultural and Topic Bias in Gener...,Biased Tales: Cultural and Topic Bias in Gener...
3,2025.emnlp-main.4,Large Language Models as Realistic Microservic...,Large Language Models as Realistic Microservic...
4,2025.emnlp-main.5,JUDGEBERT : Assessing Legal Meaning Preservati...,JUDGEBERT : Assessing Legal Meaning Preservati...


## 6. Create the cleaned corpus

For the initial corpus map, keep only papers with a paper ID, title, and abstract, and remove duplicate paper IDs.

In [8]:
df_clean = (
    df
    .drop_duplicates(subset="paper_id")
    .dropna(subset=["paper_id", "title", "abstract"])
    .copy()
)

print("Original papers:", len(df))
print("Usable papers:", len(df_clean))
print("Removed:", len(df) - len(df_clean))


Original papers: 1809
Usable papers: 1809
Removed: 0


## 7. Save cleaned dataset

The cleaned CSV becomes the input for Step 2: semantic embeddings and clustering.

In [9]:
OUTPUT_FILE = Path("../data/processed/emnlp_2025_clean.csv")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_FILE, index=False)

print("Saved to:", OUTPUT_FILE)


Saved to: ../data/processed/emnlp_2025_clean.csv


## 8. Final audit summary

Run this cell and keep the output for your project notes or README.

In [10]:
print("=== CORPUS AUDIT SUMMARY ===")
print(f"Original rows: {len(df)}")
print(f"Clean rows: {len(df_clean)}")
print(f"Removed rows: {len(df) - len(df_clean)}")
print(f"Duplicate paper IDs: {df['paper_id'].duplicated().sum()}")
print(f"Duplicate titles: {df['title'].duplicated().sum()}")
print(f"Missing abstracts: {df['abstract'].isna().sum()}")
print(f"Median abstract length: {df['abstract_word_count'].median():.0f} words")


=== CORPUS AUDIT SUMMARY ===
Original rows: 1809
Clean rows: 1809
Removed rows: 0
Duplicate paper IDs: 0
Duplicate titles: 0
Missing abstracts: 0
Median abstract length: 159 words
